In [1]:
# make sure ollama is running: ollama run codellama:7b
import os
os.environ["OLLAMA_HOST"] = "http://127.0.0.1:11434"

In [2]:
import asyncio
import ast
import random
import ollama
from collections import deque

In [3]:
# bfs traversal -- raises on bad inputs so we can catch failures
def bfs(graph, start="A"):
    if not isinstance(graph, dict):
        raise ValueError("invalid graph structure")
    if start not in graph:
        raise KeyError(f"start node '{start}' not found")

    visited = set()
    queue = deque([start])
    order = []

    while queue:
        node = queue.popleft()
        if node in visited:
            continue
        if node not in graph:
            raise KeyError(f"node '{node}' missing in graph")

        visited.add(node)
        order.append(node)

        neighbors = graph[node]
        if not isinstance(neighbors, list):
            raise ValueError(f"neighbors of '{node}' must be a list")

        for neighbor in neighbors:
            queue.append(neighbor)

    return order

In [5]:
# one valid example to anchor the llm output format
VALID_1_SHOT_EXAMPLE = '{"A": ["B", "C"], "B": ["D"], "C": [], "D": []}'

# all six failure types from the rubric
GRAPH_TYPES = [
    "valid",
    "missing_node",    # node referenced but not defined
    "cycle",           # graph contains a cycle
    "self_loop",       # node points to itself
    "disconnected",    # some nodes unreachable from start
    "invalid_structure", # values are not lists (e.g. None or ints)
    "empty",           # empty dict {}
]

# extended prompt -- describes each type so the llm knows what to generate
def extraction_prompt(graph_type: str, min_nodes: int = 3, max_nodes: int = 6) -> str:
    num_nodes = random.randint(min_nodes, max_nodes)

    type_definitions = {
        "valid":             "a correct connected graph with no missing nodes, no cycles, fully connected",
        "missing_node":      "a graph where at least one neighbor is referenced but not defined as a key",
        "cycle":             "a graph where following edges leads back to a previously visited node (e.g. A->B->A)",
        "self_loop":         "a graph where at least one node lists itself as a neighbor (e.g. A->A)",
        "disconnected":      "a graph with isolated nodes unreachable from the start node",
        "invalid_structure": "a graph where at least one value is None or an integer instead of a list",
        "empty":             "an empty dictionary with no nodes at all: {}",
    }

    definition = type_definitions.get(graph_type, graph_type)

    # empty graphs have no nodes, so skip the node count instruction
    node_instruction = f"with EXACTLY {num_nodes} nodes" if graph_type != "empty" else ""

    prompt = f"""Generate a BFS input graph in Python as an adjacency list.

Here is a valid example:
{VALID_1_SHOT_EXAMPLE}

Now generate ONE graph {node_instruction}.

The graph MUST represent this failure type: {graph_type}
Definition: {definition}

Rules:
- Use string node names like "A", "B", etc.
- Output MUST be a valid Python dictionary (except for invalid_structure and empty)
- Do NOT include explanations or comments
- Return ONLY the dictionary

Output:"""

    return prompt.strip()

In [6]:
# system prompt -- keeps llm focused on output format
system_prompt = """You are an expert in graph data structures. Generate BFS graph inputs in strict Python dict format.
Output ONLY a Python dictionary. No explanations, no markdown, no code fences.
Adjacency list format: { "A": ["B", "C"], ... }
Follow the requested failure type exactly."""

In [7]:
MODEL_NAME = "codellama:7b"
TEMPERATURE = 0.7

# async llm call with timeout so hung requests don't block forever
async def call_llm_async(prompt: str, timeout: int = 30) -> str:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": prompt}
    ]

    async def _call():
        response = await ollama.AsyncClient().chat(
            model=MODEL_NAME,
            messages=messages,
            options={"temperature": TEMPERATURE}
        )
        content = (response.get("message", {}).get("content", "") or "").strip()
        return content if content else "ERROR: empty response"

    try:
        return await asyncio.wait_for(_call(), timeout=timeout)
    except asyncio.TimeoutError:
        return "TIMEOUT"
    except Exception as e:
        return f"ERROR: {e}"

In [8]:
# try to parse the llm output as a python dict
def parse_graph(graph_text: str):
    # strip markdown code fences if llm adds them anyway
    cleaned = graph_text.strip().removeprefix("```python").removeprefix("```").removesuffix("```").strip()
    try:
        return ast.literal_eval(cleaned), "ok"
    except Exception as e:
        return None, f"parse error: {e}"

In [9]:
# classify what actually went wrong when bfs ran on the graph
def classify_failure(graph, bfs_error: str | None, graph_type: str) -> str:
    if graph is None:
        return "llm_parse_failure"
    if graph == {}:
        return "empty"
    # check for invalid structure -- any value that is not a list
    for v in graph.values():
        if not isinstance(v, list):
            return "invalid_structure"
    # check for self-loop -- any node lists itself
    for node, neighbors in graph.items():
        if isinstance(neighbors, list) and node in neighbors:
            return "self_loop"
    # check for missing node -- neighbor not in keys
    keys = set(graph.keys())
    for neighbors in graph.values():
        if isinstance(neighbors, list):
            for n in neighbors:
                if n not in keys:
                    return "missing_node"
    # check for cycle using dfs
    def has_cycle(g):
        visited, rec_stack = set(), set()
        def dfs(node):
            visited.add(node)
            rec_stack.add(node)
            for nb in (g.get(node) or []):
                if nb not in visited:
                    if dfs(nb): return True
                elif nb in rec_stack:
                    return True
            rec_stack.discard(node)
            return False
        for node in g:
            if node not in visited:
                if dfs(node): return True
        return False
    if has_cycle(graph):
        return "cycle"
    # check disconnected -- bfs from first node doesn't reach all nodes
    start = list(graph.keys())[0]
    try:
        reached = set(bfs(graph, start))
        if reached != set(graph.keys()):
            return "disconnected"
    except Exception:
        pass
    return "valid"


# run bfs and record result
def run_bfs_test(graph, graph_type: str) -> dict:
    result = {"graph": graph, "requested_type": graph_type, "behavior": None, "error": None}
    try:
        if not graph:
            raise ValueError("empty graph")
        start = list(graph.keys())[0]
        result["traversal"] = bfs(graph, start)
        result["behavior"] = "ok"
    except Exception as e:
        result["behavior"] = "error"
        result["error"] = str(e)
    result["classified_type"] = classify_failure(graph, result["error"], graph_type)
    return result

In [10]:
# generate 2 graphs per failure type (12 total, 6 unique types)
# retries up to 3x if the llm returns the wrong type
TARGETS = [
    "missing_node",
    "missing_node",
    "cycle",
    "cycle",
    "self_loop",
    "self_loop",
    "disconnected",
    "disconnected",
    "invalid_structure",
    "invalid_structure",
    "empty",
    "empty",
]

MAX_RETRIES = 3

async def generate_all(targets):
    results = []
    for i, gtype in enumerate(targets):
        print(f"generating {i+1}/{len(targets)}: {gtype}")
        # retry if llm returns wrong type
        for attempt in range(MAX_RETRIES):
            prompt = extraction_prompt(gtype)
            raw = await call_llm_async(prompt)
            graph, parse_status = parse_graph(raw)
            record = run_bfs_test(graph, gtype)
            record["raw"] = raw
            record["parse_status"] = parse_status
            classified = record["classified_type"]
            if classified == gtype or attempt == MAX_RETRIES - 1:
                results.append(record)
                suffix = f" (attempt {attempt+1})" if attempt > 0 else ""
                print(f"\tparse: {parse_status} | behavior: {record['behavior']} | classified: {classified}{suffix}")
                break
            print(f"\tretry {attempt+1}: got '{classified}', want '{gtype}'")
    return results

results = await generate_all(TARGETS)

generating 1/12: missing_node
	retry 1: got 'valid', want 'missing_node'
	parse: ok | behavior: error | classified: missing_node (attempt 2)
generating 2/12: missing_node
	parse: ok | behavior: error | classified: missing_node
generating 3/12: cycle
	parse: ok | behavior: ok | classified: cycle
generating 4/12: cycle
	parse: ok | behavior: ok | classified: cycle
generating 5/12: self_loop
	parse: ok | behavior: ok | classified: self_loop
generating 6/12: self_loop
	parse: ok | behavior: ok | classified: self_loop
generating 7/12: disconnected
	retry 1: got 'valid', want 'disconnected'
	parse: ok | behavior: ok | classified: disconnected (attempt 2)
generating 8/12: disconnected
	retry 1: got 'valid', want 'disconnected'
	parse: ok | behavior: ok | classified: disconnected (attempt 2)
generating 9/12: invalid_structure
	retry 1: got 'valid', want 'invalid_structure'
	parse: ok | behavior: ok | classified: invalid_structure (attempt 2)
generating 10/12: invalid_structure
	parse: ok | beh

In [11]:
# print a summary of each result
print("\n--- results summary ---")
for i, r in enumerate(results):
    print(f"\n[{i+1}] requested: {r['requested_type']}")
    print(f"\tclassified: {r['classified_type']}")
    print(f"\tbehavior:   {r['behavior']}")
    if r['error']:
        print(f"\terror:      {r['error']}")
    print(f"\tgraph:      {r['graph']}")


--- results summary ---

[1] requested: missing_node
	classified: missing_node
	behavior:   error
	error:      "node 'E' missing in graph"
	graph:      {'A': ['B'], 'B': ['C'], 'C': ['D'], 'D': ['E'], 'F': []}

[2] requested: missing_node
	classified: missing_node
	behavior:   error
	error:      "node 'D' missing in graph"
	graph:      {'A': ['B', 'C'], 'B': [], 'C': ['D']}

[3] requested: cycle
	classified: cycle
	behavior:   ok
	graph:      {'A': ['B'], 'B': ['C'], 'C': ['D'], 'D': ['A']}

[4] requested: cycle
	classified: cycle
	behavior:   ok
	graph:      {'A': ['B', 'C'], 'B': ['D'], 'C': ['A'], 'D': []}

[5] requested: self_loop
	classified: self_loop
	behavior:   ok
	graph:      {'A': ['A'], 'B': ['B'], 'C': ['C'], 'D': ['D'], 'E': ['E']}

[6] requested: self_loop
	classified: self_loop
	behavior:   ok
	graph:      {'A': ['B', 'C', 'A'], 'B': ['D'], 'C': [], 'D': ['A']}

[7] requested: disconnected
	classified: disconnected
	behavior:   ok
	graph:      {'A': ['B'], 'B': [], 'C'

In [12]:
# count unique failure types and check we meet the rubric requirements
from collections import Counter

classified = [r["classified_type"] for r in results]
counts = Counter(classified)

total_failures = sum(1 for r in results if r["classified_type"] != "valid")
unique_failure_types = len([k for k in counts if k not in ("valid", "llm_parse_failure")])

print("--- failure type breakdown ---")
for ftype, count in counts.most_common():
    print(f"\t{ftype}: {count}")

print(f"\ntotal failures:      {total_failures}  (need >= 10)")
print(f"unique failure types: {unique_failure_types}  (need >= 5)")
print(f"\nrubric check:")
print(f"\t>= 10 failures: {'pass' if total_failures >= 10 else 'FAIL'}")
print(f"\t>= 5 unique types: {'pass' if unique_failure_types >= 5 else 'FAIL'}")

--- failure type breakdown ---
	missing_node: 2
	cycle: 2
	self_loop: 2
	disconnected: 2
	invalid_structure: 2
	empty: 2

total failures:      12  (need >= 10)
unique failure types: 6  (need >= 5)

rubric check:
	>= 10 failures: pass
	>= 5 unique types: pass
